In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
df_base = pd.read_excel('data_validacion.xlsx')
df_calidad = pd.read_excel('data_lab.xlsx')

In [3]:
df_calidad['HORA_PRODUCCION'] = df_calidad['HORA_PRODUCCION'].str.replace('a. m.','AM').str.replace('p. m.','PM')

C:\Users\sh0371c\AppData\Local\Temp\ipykernel_22284\71878994.py:1: FutureWarning: The default value of regex will change from True to False in a future version.
  df_calidad['HORA_PRODUCCION'] = df_calidad['HORA_PRODUCCION'].str.replace('a. m.','AM').str.replace('p. m.','PM')


In [4]:
df_calidad['HORA_PRODUCCION']=pd.to_datetime(df_calidad['HORA_PRODUCCION'],format='%d/%m/%Y %I:%M:%S %p',errors='coerce',dayfirst=True)

In [5]:
def encontrar_cenizas_lab(timestamp):
    diferencia = np.abs(df_calidad['HORA_PRODUCCION'] - timestamp)
    indice_min = diferencia.idxmin()
    return df_calidad.loc[indice_min, 'CENIZAS LAB']


In [6]:
df_base['CENIZAS_LAB1'] = df_base['Timestamp'].apply(encontrar_cenizas_lab)

In [7]:
df_base.to_excel('data_validacion_final.xlsx',index=False)

In [8]:
df_base.rename({'4dw1.ctrl:MV':'4dw1.ctrl'},axis=1,inplace=True)

In [9]:
features = ['42nic073', '42nic025', '42fic109', '44fic108', '44dic108', '4.kgtr.agret',
            '42nt122.b', 'cenizas_total', 'ret_1er_paso_mv', '4dw1.ctrl']
target = 'CENIZAS_LAB1'

#df_base = df_base.drop('Timestamp', axis=1)

X = df_base[features]
y = df_base[target]

In [10]:
import joblib

# cargar el modelo entrenado
model = joblib.load("modelo_random_forest_V1.pkl")

In [11]:
y_pred = model.predict(X)

In [12]:
df_base['prediction']=y_pred.tolist()

In [14]:
df_base.to_excel('prediccion_vs_real.xlsx',index=False)